# Multivariate TimeSeries Prediction

This dataset will be processed to include only numeric values, and will be used in DL models for "multivariate time series prediction" tasks.

This means that the Deep Learning models will learn from it and predict more than one target.

The time series data cannot be shuffled during training, because the order of the data matters.

In [1]:
# Gerekli kütüphaneleri yükleme
# Eğer yüklü değilse: pip install torch torchvision matplotlib numpy

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms

# Reproducibility için seed ayarla
torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch Versiyonu : {torch.__version__}")
print(f"GPU Kullanılabilir : {torch.cuda.is_available()}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Kullanılan Cihaz  : {device}")

PyTorch Versiyonu : 2.8.0+cpu
GPU Kullanılabilir : False
Kullanılan Cihaz  : cpu


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd

## Feature Description

| Index | Features         | Format            | Description                                                                                                                                                                                                            |
|-------|------------------|-------------------|------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| 1     | Date Time        | 01.01.2009 00:10:00 | Date-time reference                                                                                                                                                                                                    |
| 2     | p (mbar)         | 996.52            | The pascal SI derived unit of pressure used to quantify internal pressure. Meteorological reports typically state atmospheric pressure in millibars.                                                                      |
| 3     | T (degC)         | -8.02             | Temperature in Celsius                                                                                                                                                                                                 |
| 4     | Tpot (K)         | 265.4             | Temperature in Kelvin                                                                                                                                                                                                  |
| 5     | Tdew (degC)      | -8.9              | Temperature in Celsius relative to humidity. Dew Point is a measure of the absolute amount of water in the air, the DP is the temperature at which the air cannot hold all the moisture in it and water condenses. |
| 6     | rh (%)           | 93.3              | Relative Humidity is a measure of how saturated the air is with water vapor, the %RH determines the amount of water contained within collection objects.                                                       |
| 7     | VPmax (mbar)     | 3.33              | Saturation vapor pressure                                                                                                                                                                                              |
| 8     | VPact (mbar)     | 3.11              | Vapor pressure                                                                                                                                                                                                         |
| 9     | VPdef (mbar)     | 0.22              | Vapor pressure deficit                                                                                                                                                                                                 |
| 10    | sh (g/kg)        | 1.94              | Specific humidity                                                                                                                                                                                                      |
| 11    | H2OC (mmol/mol)  | 3.12              | Water vapor concentration                                                                                                                                                                                              |
| 12    | rho (g/m ** 3)   | 1307.75           | Airtight                                                                                                                                                                                                               |
| 13    | wv (m/s)         | 1.03              | Wind speed                                                                                                                                                                                                             |
| 14    | max. wv (m/s)    | 1.75              | Maximum wind speed                                                                                                                                                                                                     |
| 15    | wd (deg)         | 152.3             | Wind direction in degrees                                                                                                                                                                                              |

In [3]:
from meta import *

df = pd.read_csv(datapath+r"\climate.csv")

In [4]:
df=df.dropna()
df=df.drop_duplicates()
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 420224 entries, 0 to 420550
Data columns (total 15 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   Date Time        420224 non-null  object 
 1   p (mbar)         420224 non-null  float64
 2   T (degC)         420224 non-null  float64
 3   Tpot (K)         420224 non-null  float64
 4   Tdew (degC)      420224 non-null  float64
 5   rh (%)           420224 non-null  float64
 6   VPmax (mbar)     420224 non-null  float64
 7   VPact (mbar)     420224 non-null  float64
 8   VPdef (mbar)     420224 non-null  float64
 9   sh (g/kg)        420224 non-null  float64
 10  H2OC (mmol/mol)  420224 non-null  float64
 11  rho (g/m**3)     420224 non-null  float64
 12  wv (m/s)         420224 non-null  float64
 13  max. wv (m/s)    420224 non-null  float64
 14  wd (deg)         420224 non-null  float64
dtypes: float64(14), object(1)
memory usage: 51.3+ MB


In [5]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
p (mbar),420224.0,989.214157,8.360888,913.60,984.20,989.58,994.73,1015.35
T (degC),420224.0,9.442421,8.421135,-23.01,3.36,9.40,15.46,37.28
Tpot (K),420224.0,283.484880,8.502206,250.60,277.43,283.46,289.52,311.34
Tdew (degC),420224.0,4.953472,6.731171,-25.01,0.23,5.21,10.07,23.11
rh (%),420224.0,76.028738,16.460467,12.95,65.24,79.30,89.40,100.00
VPmax (mbar),420224.0,13.568642,7.734770,0.95,7.77,11.81,17.59,63.77
VPact (mbar),420224.0,9.532333,4.183996,0.79,6.21,8.86,12.35,28.32
VPdef (mbar),420224.0,4.036225,4.891287,0.00,0.87,2.18,5.29,46.01
sh (g/kg),420224.0,6.021503,2.656043,0.50,3.92,5.59,7.80,18.13
H2OC (mmol/mol),420224.0,9.638778,4.235244,0.80,6.28,8.96,12.48,28.82


In [6]:
df.sample(5)

,Date Time,p (mbar),T (degC),Tpot (K),Tdew (degC),rh (%),VPmax (mbar),VPact (mbar),VPdef (mbar),sh (g/kg),H2OC (mmol/mol),rho (g/m**3),wv (m/s),max. wv (m/s),wd (deg)
412989,09.11.2016 11:50:00,979.86,3.46,278.23,-0.83,73.40,7.83,5.75,2.08,3.66,5.87,1231.26,3.91,5.93,171.50
347151,07.08.2015 04:40:00,989.02,19.21,293.29,13.23,68.28,22.30,15.23,7.07,9.63,15.39,1171.56,0.32,0.88,80.30
80834,15.07.2010 08:50:00,988.11,21.48,295.65,16.00,71.00,25.65,18.21,7.44,11.55,18.43,1160.13,2.19,3.48,223.40
292828,25.07.2014 06:50:00,987.65,15.60,289.79,14.82,95.10,17.75,16.88,0.87,10.70,17.09,1183.80,0.61,1.04,240.40
238429,13.07.2013 18:50:00,993.61,22.27,295.97,11.37,50.03,26.92,13.47,13.45,8.47,13.55,1165.62,1.12,3.42,5.74


In [7]:
df["Date Time"] = pd.to_datetime(df["Date Time"], format="%d.%m.%Y %H:%M:%S")

In [8]:
df = df.sort_values("Date Time")

In [9]:
# Extract time features
df["hour"] = df["Date Time"].dt.hour
df["day"] = df["Date Time"].dt.day
df["month"] = df["Date Time"].dt.month
df["dayofweek"] = df["Date Time"].dt.dayofweek
df["dayofyear"] = df["Date Time"].dt.dayofyear

# Cyclical encoding
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

df["day_sin"] = np.sin(2 * np.pi * df["day"] / 31)
df["day_cos"] = np.cos(2 * np.pi * df["day"] / 31)

df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

df["dayofweek_sin"] = np.sin(2 * np.pi * df["dayofweek"] / 7)
df["dayofweek_cos"] = np.cos(2 * np.pi * df["dayofweek"] / 7)

df["dayofyear_sin"] = np.sin(2 * np.pi * df["dayofyear"] / 365.25)
df["dayofyear_cos"] = np.cos(2 * np.pi * df["dayofyear"] / 365.25)

# Drop raw calendar columns and datetime
df.drop(
    columns=[
        "Date Time",
        "hour",
        "day",
        "month",
        "dayofweek",
        "dayofyear",
    ],
    inplace=True,
)

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 420224 entries, 0 to 420550
Data columns (total 24 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   p (mbar)         420224 non-null  float64
 1   T (degC)         420224 non-null  float64
 2   Tpot (K)         420224 non-null  float64
 3   Tdew (degC)      420224 non-null  float64
 4   rh (%)           420224 non-null  float64
 5   VPmax (mbar)     420224 non-null  float64
 6   VPact (mbar)     420224 non-null  float64
 7   VPdef (mbar)     420224 non-null  float64
 8   sh (g/kg)        420224 non-null  float64
 9   H2OC (mmol/mol)  420224 non-null  float64
 10  rho (g/m**3)     420224 non-null  float64
 11  wv (m/s)         420224 non-null  float64
 12  max. wv (m/s)    420224 non-null  float64
 13  wd (deg)         420224 non-null  float64
 14  hour_sin         420224 non-null  float64
 15  hour_cos         420224 non-null  float64
 16  day_sin          420224 non-null  float64
 

In [11]:
df.to_csv(datapath+r"\climate_prep.csv", index=False, encoding="utf-8")